# 👁️ How Computers See: CNNs & Spatial Intelligence
## AI Understands SPACE

**Workshop 2: Foundations of AI - Deep Learning & Computer Vision**

In this notebook, we'll explore:
- How images are just grids of numbers
- Convolution: the operation that gives AI spatial awareness
- How CNNs build understanding from edges → shapes → objects

In [ ]:
# =============================================================================
# PART 1: IMAGES ARE JUST NUMBERS
# =============================================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

import plotly.io as pio
pio.renderers.default = "notebook"

display(HTML("""
<div class="concept-box">
    <h2 style="margin-top:0; color:#11998e;">🔢 The Digital Reality: Images = Numbers</h2>
    <p>What you see as a beautiful image, a computer sees as a <strong>massive grid of numbers</strong>.</p>
    <p>Each pixel is just 3 values: <span style="color:#FF6B6B;">Red</span>, 
       <span style="color:#4CAF50;">Green</span>, <span style="color:#2196F3;">Blue</span> (0-255)</p>
</div>
"""))

def create_pixel_explorer():
    """Interactive visualization showing how images are made of pixels"""
    
    # Create a simple 8x8 "face" image for demonstration
    face = np.zeros((8, 8, 3), dtype=np.uint8)
    
    # Background - light skin tone
    face[:, :] = [255, 220, 185]
    
    # Eyes - dark
    face[2, 2] = [50, 50, 50]
    face[2, 5] = [50, 50, 50]
    
    # Nose
    face[4, 3:5] = [200, 160, 140]
    
    # Smile
    face[5, 2] = [200, 100, 100]
    face[6, 3:5] = [200, 100, 100]
    face[5, 5] = [200, 100, 100]
    
    # Hair
    face[0, :] = [80, 50, 30]
    face[1, 0] = [80, 50, 30]
    face[1, 7] = [80, 50, 30]
    
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            '<b>What You See</b>',
            '<b>What the Computer Sees</b>',
            '<b>One Pixel Up Close</b>'
        ),
        column_widths=[0.3, 0.4, 0.3],
        horizontal_spacing=0.08
    )
    
    # 1. Visual representation (what humans see)
    fig.add_trace(go.Heatmap(
        z=np.mean(face, axis=2)[::-1],
        colorscale=[[0, 'rgb(80,50,30)'], [0.5, 'rgb(200,160,140)'], [1, 'rgb(255,220,185)']],
        showscale=False,
        hoverinfo='skip'
    ), row=1, col=1)
    
    # Add grid lines for pixels
    for i in range(9):
        fig.add_shape(type='line', x0=i-0.5, y0=-0.5, x1=i-0.5, y1=7.5,
                     line=dict(color='rgba(0,0,0,0.3)', width=1), row=1, col=1)
        fig.add_shape(type='line', x0=-0.5, y0=i-0.5, x1=7.5, y1=i-0.5,
                     line=dict(color='rgba(0,0,0,0.3)', width=1), row=1, col=1)
    
    # 2. Number grid representation
    # Create heatmap with numbers
    z_values = np.mean(face, axis=2)[::-1]
    
    # Create text annotations for each pixel
    annotations_text = []
    for i in range(8):
        row_text = []
        for j in range(8):
            r, g, b = face[7-i, j]
            row_text.append(f'{r},{g},{b}')
        annotations_text.append(row_text)
    
    fig.add_trace(go.Heatmap(
        z=z_values,
        text=annotations_text,
        texttemplate='<b>%{text}</b>',
        textfont=dict(size=8, color='black'),
        colorscale='Greys',
        showscale=False,
        hovertemplate='Row: %{y}<br>Col: %{x}<br>RGB: %{text}<extra></extra>'
    ), row=1, col=2)
    
    # 3. Single pixel breakdown (showing the eye pixel)
    eye_pixel = face[2, 2]  # The eye
    
    # RGB bars
    colors = ['#FF6B6B', '#4CAF50', '#2196F3']
    labels = ['Red', 'Green', 'Blue']
    values = [eye_pixel[0], eye_pixel[1], eye_pixel[2]]
    
    fig.add_trace(go.Bar(
        x=labels,
        y=values,
        marker_color=colors,
        text=[str(v) for v in values],
        textposition='outside',
        textfont=dict(size=14, color='black'),
        showlegend=False,
        hovertemplate='%{x}: %{y}<extra></extra>'
    ), row=1, col=3)
    
    # Add the actual color as a colored rectangle annotation
    fig.add_annotation(
        x=1, y=200,
        text=f'<b>Pixel Color</b><br>RGB({eye_pixel[0]}, {eye_pixel[1]}, {eye_pixel[2]})',
        showarrow=False,
        font=dict(size=12),
        bgcolor=f'rgb({eye_pixel[0]},{eye_pixel[1]},{eye_pixel[2]})',
        bordercolor='black',
        borderwidth=2,
        borderpad=10,
        row=1, col=3
    )
    
    fig.update_layout(
        height=400,
        width=1100,
        showlegend=False,
        paper_bgcolor='white',
        plot_bgcolor='rgba(248, 249, 250, 0.5)',
        margin=dict(l=40, r=40, t=60, b=40)
    )
    
    fig.update_xaxes(visible=False, row=1, col=1)
    fig.update_yaxes(visible=False, row=1, col=1)
    fig.update_xaxes(visible=False, row=1, col=2)
    fig.update_yaxes(visible=False, row=1, col=2)
    fig.update_xaxes(title='Channel', row=1, col=3)
    fig.update_yaxes(title='Value (0-255)', range=[0, 280], row=1, col=3)
    
    return fig

fig = create_pixel_explorer()
fig.show()

display(HTML("""
<div class="insight-box">
    <h3 style="margin-top:0; color:#856404;">💡 Scale This Up!</h3>
    <p>A simple smartphone photo (12 megapixels) = <strong>36 MILLION numbers</strong></p>
    <p>That's 12,000,000 pixels × 3 color channels (RGB)</p>
    <p style="margin-bottom:0;"><em>How does AI make sense of millions of numbers? That's where CNNs come in!</em></p>
</div>
"""))

In [ ]:
# =============================================================================
# PART 2: ANIMATED CONVOLUTION - WATCH THE KERNEL SLIDE!
# =============================================================================

display(HTML("""
<div class="concept-box">
    <h2 style="margin-top:0; color:#11998e;">🔍 Convolution: How AI "Scans" Images</h2>
    <p>The key insight: <strong>Local patterns matter!</strong></p>
    <p>A small "filter" (kernel) slides across the image step by step, looking for specific patterns like edges.</p>
</div>
"""))

class AnimatedConvolutionViz:
    """Interactive visualization showing a kernel sliding over an image"""
    
    def __init__(self):
        # Create a more interesting test image (simple shapes)
        self.image_size = 10
        self.kernel_size = 3
        self.image = self.create_sample_image()
        
        # Define filters with clear purposes
        self.filters = {
            'Horizontal Edge': {
                'kernel': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float),
                'description': 'Detects horizontal edges by comparing pixels above vs below',
                'color': '#FF6B6B'
            },
            'Vertical Edge': {
                'kernel': np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=float),
                'description': 'Detects vertical edges by comparing pixels left vs right',
                'color': '#4ECDC4'
            },
            'Diagonal Edge': {
                'kernel': np.array([[2, 1, 0], [1, 0, -1], [0, -1, -2]], dtype=float),
                'description': 'Detects diagonal edges from top-left to bottom-right',
                'color': '#45B7D1'
            },
            'Sharpen': {
                'kernel': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float),
                'description': 'Enhances edges and details by emphasizing the center',
                'color': '#96CEB4'
            },
            'Blur': {
                'kernel': np.array([[1, 1, 1], [1, 1, 1], [1, 1, 1]], dtype=float) / 9,
                'description': 'Smooths the image by averaging neighboring pixels',
                'color': '#DDA0DD'
            }
        }
        
        self.current_filter = 'Horizontal Edge'
        self.positions = self._generate_positions()
        self.is_playing = False
        
        self.create_interface()
    
    def create_sample_image(self):
        """Create a test image with clear geometric shapes"""
        img = np.zeros((self.image_size, self.image_size))
        
        # Background gradient
        img[:, :] = 50
        
        # Bright rectangle (creates horizontal and vertical edges)
        img[2:8, 2:8] = 200
        
        # Inner square (more edges)
        img[4:6, 4:6] = 255
        
        # Add diagonal line
        for i in range(self.image_size):
            if i < self.image_size:
                img[i, min(i, self.image_size-1)] = max(img[i, min(i, self.image_size-1)], 150)
        
        return img
    
    def _generate_positions(self):
        """Generate all valid kernel positions"""
        positions = []
        output_size = self.image_size - self.kernel_size + 1
        for i in range(output_size):
            for j in range(output_size):
                positions.append((i, j))
        return positions
    
    def apply_convolution_full(self, kernel):
        """Apply full convolution to get complete output"""
        output_size = self.image_size - self.kernel_size + 1
        output = np.zeros((output_size, output_size))
        
        for i in range(output_size):
            for j in range(output_size):
                region = self.image[i:i+self.kernel_size, j:j+self.kernel_size]
                output[i, j] = np.sum(region * kernel)
        
        return output
    
    def create_interface(self):
        """Create the interactive interface"""
        
        # Header
        self.header = widgets.HTML("""
        <div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); 
                    color: white; padding: 20px; border-radius: 12px; text-align: center;
                    margin-bottom: 15px; font-family: 'Inter', sans-serif;">
            <h3 style="margin: 0;">🎬 Watch Convolution in Action</h3>
            <p style="margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;">
                Step through to see how the filter slides across the image detecting features
            </p>
        </div>
        """)
        
        # Filter selector
        self.filter_dropdown = widgets.Dropdown(
            options=list(self.filters.keys()),
            value='Horizontal Edge',
            description='Filter:',
            style={'description_width': '50px'},
            layout=widgets.Layout(width='200px')
        )
        
        # Position slider
        self.position_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=len(self.positions) - 1,
            step=1,
            description='Step:',
            style={'description_width': '50px'},
            layout=widgets.Layout(width='400px'),
            continuous_update=False
        )
        
        # Play/Pause button
        self.play_btn = widgets.Button(
            description='▶️ Play',
            button_style='success',
            layout=widgets.Layout(width='100px')
        )
        
        # Reset button
        self.reset_btn = widgets.Button(
            description='🔄 Reset',
            button_style='warning',
            layout=widgets.Layout(width='100px')
        )
        
        # Info display
        self.info_display = widgets.HTML()
        self.calculation_display = widgets.HTML()
        
        # Plot output
        self.plot_output = widgets.Output()
        
        # Connect events
        self.filter_dropdown.observe(self.on_filter_change, 'value')
        self.position_slider.observe(self.update_visualization, 'value')
        self.play_btn.on_click(self.play_animation)
        self.reset_btn.on_click(self.reset)
        
        # Layout
        controls = widgets.HBox([
            self.filter_dropdown,
            self.position_slider,
            self.play_btn,
            self.reset_btn
        ], layout=widgets.Layout(justify_content='center', gap='15px', margin='10px 0'))
        
        display(self.header)
        display(controls)
        display(self.info_display)
        display(self.plot_output)
        display(self.calculation_display)
        
        # Initial visualization
        self.update_visualization(None)
    
    def on_filter_change(self, change):
        self.current_filter = change['new']
        self.update_visualization(None)
    
    def update_visualization(self, change):
        """Update the visualization for current position"""
        pos_idx = self.position_slider.value
        row, col = self.positions[pos_idx]
        
        filter_info = self.filters[self.current_filter]
        kernel = filter_info['kernel']
        
        # Get the region under the kernel
        region = self.image[row:row+self.kernel_size, col:col+self.kernel_size]
        
        # Calculate output value
        output_value = np.sum(region * kernel)
        
        # Get full output for comparison
        full_output = self.apply_convolution_full(kernel)
        
        # Update info display
        self.info_display.value = f"""
        <div style="background: linear-gradient(135deg, #e8f5e9 0%, #c8e6c9 100%); 
                    padding: 15px; border-radius: 10px; margin: 10px 0; text-align: center;
                    border-left: 4px solid {filter_info['color']};">
            <strong style="color: {filter_info['color']};">{self.current_filter}:</strong> 
            {filter_info['description']}
            <br><br>
            <span style="font-size: 16px;">📍 Position: ({row}, {col}) | 
            Step {pos_idx + 1} of {len(self.positions)}</span>
        </div>
        """
        
        with self.plot_output:
            clear_output(wait=True)
            
            fig = make_subplots(
                rows=2, cols=3,
                subplot_titles=(
                    '<b>📷 Input Image</b><br><span style="font-size:11px">Kernel scanning position shown in red</span>',
                    '<b>🔲 Filter (Kernel)</b><br><span style="font-size:11px">The pattern detector</span>',
                    '<b>📊 Output Feature Map</b><br><span style="font-size:11px">Detected features light up</span>',
                    '<b>🔍 Region Under Kernel</b>',
                    '<b>✖️ Element-wise Multiply</b>',
                    '<b>➕ Result</b>'
                ),
                vertical_spacing=0.15,
                horizontal_spacing=0.08,
                row_heights=[0.55, 0.45]
            )
            
            # === ROW 1: Main visualization ===
            
            # 1. Input image with kernel position highlighted
            fig.add_trace(go.Heatmap(
                z=self.image[::-1],
                colorscale='Greys',
                showscale=False,
                hovertemplate='Row: %{y}<br>Col: %{x}<br>Value: %{z}<extra></extra>'
            ), row=1, col=1)
            
            # Draw kernel position rectangle
            kernel_row = self.image_size - 1 - row - self.kernel_size + 1
            fig.add_shape(
                type='rect',
                x0=col - 0.5, y0=kernel_row - 0.5,
                x1=col + self.kernel_size - 0.5, y1=kernel_row + self.kernel_size - 0.5,
                line=dict(color=filter_info['color'], width=4),
                fillcolor='rgba(255, 107, 107, 0.2)',
                row=1, col=1
            )
            
            # Add grid
            for i in range(self.image_size + 1):
                fig.add_shape(type='line', x0=i-0.5, y0=-0.5, x1=i-0.5, y1=self.image_size-0.5,
                             line=dict(color='rgba(0,0,0,0.15)', width=0.5), row=1, col=1)
                fig.add_shape(type='line', x0=-0.5, y0=i-0.5, x1=self.image_size-0.5, y1=i-0.5,
                             line=dict(color='rgba(0,0,0,0.15)', width=0.5), row=1, col=1)
            
            # 2. Kernel visualization
            kernel_text = [[f'{v:.1f}' for v in k_row] for k_row in kernel]
            fig.add_trace(go.Heatmap(
                z=kernel[::-1],
                colorscale=[[0, '#FF6B6B'], [0.5, '#FFFFFF'], [1, '#4CAF50']],
                zmid=0,
                showscale=False,
                text=kernel_text[::-1],
                texttemplate='<b>%{text}</b>',
                textfont=dict(size=16, color='black'),
                hovertemplate='Value: %{z:.2f}<extra></extra>'
            ), row=1, col=2)
            
            # 3. Output feature map (partial, showing progress)
            output_size = self.image_size - self.kernel_size + 1
            output_partial = np.full((output_size, output_size), np.nan)
            
            # Fill in computed values up to current position
            for idx in range(pos_idx + 1):
                r, c = self.positions[idx]
                output_partial[r, c] = full_output[r, c]
            
            # Normalize for display
            vmax = max(abs(np.nanmin(full_output)), abs(np.nanmax(full_output)))
            
            fig.add_trace(go.Heatmap(
                z=output_partial[::-1],
                colorscale=[[0, '#3b1f71'], [0.5, '#1a1a2e'], [1, '#FFD700']],
                zmin=-vmax if vmax > 0 else -1,
                zmax=vmax if vmax > 0 else 1,
                showscale=True,
                colorbar=dict(title='Activation', len=0.4, y=0.78),
                hovertemplate='Row: %{y}<br>Col: %{x}<br>Activation: %{z:.1f}<extra></extra>'
            ), row=1, col=3)
            
            # Highlight current output position
            out_row = output_size - 1 - row
            fig.add_shape(
                type='rect',
                x0=col - 0.5, y0=out_row - 0.5,
                x1=col + 0.5, y1=out_row + 0.5,
                line=dict(color='#FF6B6B', width=3),
                row=1, col=3
            )
            
            # === ROW 2: Step-by-step calculation ===
            
            # 4. Region under kernel
            region_text = [[f'{int(v)}' for v in r_row] for r_row in region]
            fig.add_trace(go.Heatmap(
                z=region[::-1],
                colorscale='Greys',
                showscale=False,
                text=region_text[::-1],
                texttemplate='<b>%{text}</b>',
                textfont=dict(size=14, color='white'),
                hoverinfo='skip'
            ), row=2, col=1)
            
            # 5. Element-wise multiplication result
            multiplication = region * kernel
            mult_text = [[f'{v:.0f}' for v in m_row] for m_row in multiplication]
            fig.add_trace(go.Heatmap(
                z=multiplication[::-1],
                colorscale=[[0, '#FF6B6B'], [0.5, '#FFFFFF'], [1, '#4CAF50']],
                zmid=0,
                showscale=False,
                text=mult_text[::-1],
                texttemplate='<b>%{text}</b>',
                textfont=dict(size=14, color='black'),
                hoverinfo='skip'
            ), row=2, col=2)
            
            # 6. Sum visualization (single value)
            sum_color = '#4CAF50' if output_value > 0 else '#FF6B6B' if output_value < 0 else '#888'
            fig.add_trace(go.Scatter(
                x=[1], y=[1],
                mode='text',
                text=[f'<b>{output_value:.1f}</b>'],
                textfont=dict(size=40, color=sum_color),
                showlegend=False,
                hoverinfo='skip'
            ), row=2, col=3)
            
            # Add sum annotation
            fig.add_annotation(
                x=1, y=0.3,
                text='<b>↑ This value goes into<br>the output feature map!</b>',
                showarrow=False,
                font=dict(size=11, color='#666'),
                row=2, col=3
            )
            
            # Update layout
            fig.update_layout(
                height=700,
                width=1100,
                showlegend=False,
                paper_bgcolor='white',
                plot_bgcolor='white',
                margin=dict(l=40, r=40, t=80, b=40)
            )
            
            # Hide axes for cleaner look
            for r in [1, 2]:
                for c in [1, 2, 3]:
                    fig.update_xaxes(visible=False, row=r, col=c)
                    fig.update_yaxes(visible=False, row=r, col=c)
            
            # Set explicit range for sum subplot to position text properly
            fig.update_xaxes(range=[0, 2], row=2, col=3)
            fig.update_yaxes(range=[0, 2], row=2, col=3)
            
            fig.show()
        
        # Update calculation display
        calc_parts = [f'{int(region[i,j])}×{kernel[i,j]:.1f}' 
                     for i in range(3) for j in range(3)]
        self.calculation_display.value = f"""
        <div style="background: #f8f9fa; padding: 15px; border-radius: 10px; 
                    margin: 10px 0; font-family: monospace; text-align: center;">
            <strong>Calculation:</strong> ({' + '.join(calc_parts)}) = <span style="color: {sum_color}; font-size: 18px;"><b>{output_value:.1f}</b></span>
        </div>
        """
    
    def play_animation(self, b):
        """Animate through all positions"""
        import time
        
        if self.is_playing:
            self.is_playing = False
            self.play_btn.description = '▶️ Play'
            return
        
        self.is_playing = True
        self.play_btn.description = '⏸️ Pause'
        
        for i in range(self.position_slider.value, len(self.positions)):
            if not self.is_playing:
                break
            self.position_slider.value = i
            time.sleep(0.15)
        
        self.is_playing = False
        self.play_btn.description = '▶️ Play'
    
    def reset(self, b):
        """Reset to start"""
        self.is_playing = False
        self.play_btn.description = '▶️ Play'
        self.position_slider.value = 0

# Create the visualization
conv_viz = AnimatedConvolutionViz()

display(HTML("""
<div class="insight-box">
    <h3 style="margin-top:0; color:#856404;">💡 What You're Seeing</h3>
    <ul style="margin-bottom: 0;">
        <li><strong>Red box</strong> on input = where the kernel is currently "looking"</li>
        <li><strong>Feature map</strong> builds up as the kernel scans → bright spots = strong feature detection!</li>
        <li><strong>Positive values</strong> (green) = pattern found! <strong>Negative values</strong> (red) = opposite pattern</li>
        <li>Try different filters to see how each detects different features!</li>
    </ul>
</div>
"""))

In [ ]:
# =============================================================================
# PART 3: CNN FEATURE HIERARCHY - INTERACTIVE EXPLORER
# =============================================================================

display(HTML("""
<div class="concept-box">
    <h2 style="margin-top:0; color:#11998e;">🏔️ The CNN Hierarchy: From Edges to Objects</h2>
    <p>The magic of CNNs: <strong>each layer builds on the previous one!</strong></p>
    <p>Early layers see edges → Middle layers combine edges into textures & parts → Deep layers recognize objects</p>
</div>
"""))

class CNNFeatureHierarchyExplorer:
    """Interactive visualization showing progressive feature learning in CNNs"""
    
    def __init__(self):
        self.layers = [
            {
                'name': 'Input Image',
                'short': 'Input',
                'color': '#4ECDC4',
                'icon': '📷',
                'learns': 'Raw pixel values',
                'example_features': ['RGB colors', 'Brightness', 'Position'],
                'description': 'The original image as the network sees it - just a grid of numbers representing colors.',
                'visual': self._create_input_visual
            },
            {
                'name': 'Layer 1: Edge Detection',
                'short': 'Edges',
                'color': '#FF6B6B',
                'icon': '📐',
                'learns': 'Simple edges & gradients',
                'example_features': ['Horizontal edges', 'Vertical edges', 'Diagonal lines', 'Color gradients'],
                'description': 'First conv layers learn to detect basic edges - the building blocks of all visual patterns.',
                'visual': self._create_edge_visual
            },
            {
                'name': 'Layer 2-3: Textures & Patterns',
                'short': 'Textures',
                'color': '#FFD93D',
                'icon': '🎨',
                'learns': 'Combinations of edges',
                'example_features': ['Fur texture', 'Stripes', 'Grids', 'Curves', 'Corners'],
                'description': 'Middle layers combine edges into more complex patterns like textures and repeated motifs.',
                'visual': self._create_texture_visual
            },
            {
                'name': 'Layer 4-5: Parts & Shapes',
                'short': 'Parts',
                'color': '#6BCB77',
                'icon': '👁️',
                'learns': 'Object components',
                'example_features': ['Eyes', 'Ears', 'Wheels', 'Windows', 'Noses'],
                'description': 'Deeper layers recognize meaningful parts of objects - eyes, wheels, windows, etc.',
                'visual': self._create_parts_visual
            },
            {
                'name': 'Deep Layers: Objects',
                'short': 'Objects',
                'color': '#9B59B6',
                'icon': '🎯',
                'learns': 'Complete objects',
                'example_features': ['Cat faces', 'Dog faces', 'Cars', 'Buildings'],
                'description': 'The deepest layers combine parts into complete object concepts ready for classification.',
                'visual': self._create_objects_visual
            },
            {
                'name': 'Output: Classification',
                'short': 'Output',
                'color': '#E74C3C',
                'icon': '✅',
                'learns': 'Final decision',
                'example_features': ['Cat: 95%', 'Dog: 3%', 'Bird: 2%'],
                'description': 'The final layer produces probability scores for each possible class.',
                'visual': self._create_output_visual
            }
        ]
        
        self.create_interface()
    
    def _create_input_visual(self):
        """Create input image visualization"""
        # Create a simple cat-like image
        img = np.zeros((20, 20))
        # Face
        for i in range(20):
            for j in range(20):
                dist = np.sqrt((i-10)**2 + (j-10)**2)
                if dist < 8:
                    img[i, j] = 200
        # Ears
        img[2:6, 3:6] = 180
        img[2:6, 14:17] = 180
        # Eyes
        img[8:11, 6:8] = 50
        img[8:11, 12:14] = 50
        # Nose
        img[12:14, 9:11] = 100
        return img, 'Greys'
    
    def _create_edge_visual(self):
        """Create edge detection visualization"""
        img, _ = self._create_input_visual()
        # Apply simple edge detection
        kernel = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
        from scipy.ndimage import convolve
        edges = convolve(img, kernel)
        return edges, [[0, '#000033'], [0.5, '#333366'], [1, '#FFFF00']]
    
    def _create_texture_visual(self):
        """Create texture detection visualization"""
        img = np.zeros((20, 20))
        # Create texture patterns
        for i in range(20):
            for j in range(20):
                # Fur-like texture
                if (i + j) % 3 == 0:
                    img[i, j] = 200
                elif (i - j) % 4 == 0:
                    img[i, j] = 150
                else:
                    img[i, j] = 80
        # Add curved regions
        for i in range(8, 15):
            for j in range(8, 15):
                img[i, j] = 255
        return img, 'YlOrRd'
    
    def _create_parts_visual(self):
        """Create parts detection visualization"""
        img = np.zeros((20, 20))
        # Eye region activated
        img[6:12, 4:9] = 200  # Left eye area
        img[6:12, 11:16] = 200  # Right eye area
        # Nose region
        img[11:15, 8:12] = 150
        return img, 'Greens'
    
    def _create_objects_visual(self):
        """Create object detection visualization"""
        img = np.zeros((20, 20))
        # Whole face activated
        for i in range(20):
            for j in range(20):
                dist = np.sqrt((i-10)**2 + (j-10)**2)
                if dist < 9:
                    img[i, j] = 255 - dist * 20
        return img, 'Purples'
    
    def _create_output_visual(self):
        """Create output visualization"""
        # Just return probabilities as a simple array
        img = np.array([[95, 3, 2]]).T
        return img, 'RdYlGn'
    
    def create_interface(self):
        """Create the interactive interface"""
        
        # Header
        display(HTML("""
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    color: white; padding: 20px; border-radius: 12px; text-align: center;
                    margin-bottom: 15px; font-family: 'Inter', sans-serif;">
            <h3 style="margin: 0;">🔬 CNN Feature Hierarchy Explorer</h3>
            <p style="margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;">
                Click through each layer to see how CNNs progressively build understanding
            </p>
        </div>
        """))
        
        # Layer selector
        self.layer_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=len(self.layers) - 1,
            step=1,
            description='Layer:',
            style={'description_width': '50px'},
            layout=widgets.Layout(width='500px'),
            continuous_update=False
        )
        
        # Navigation buttons
        self.prev_btn = widgets.Button(
            description='◀️ Previous',
            button_style='info',
            layout=widgets.Layout(width='110px')
        )
        self.next_btn = widgets.Button(
            description='Next ▶️',
            button_style='info',
            layout=widgets.Layout(width='110px')
        )
        
        # Info display
        self.layer_info = widgets.HTML()
        
        # Plot output
        self.plot_output = widgets.Output()
        
        # Connect events
        self.layer_slider.observe(self.update_visualization, 'value')
        self.prev_btn.on_click(lambda b: self._change_layer(-1))
        self.next_btn.on_click(lambda b: self._change_layer(1))
        
        # Layout
        nav_controls = widgets.HBox([
            self.prev_btn,
            self.layer_slider,
            self.next_btn
        ], layout=widgets.Layout(justify_content='center', gap='15px', margin='10px 0'))
        
        display(nav_controls)
        display(self.layer_info)
        display(self.plot_output)
        
        # Initial visualization
        self.update_visualization(None)
    
    def _change_layer(self, delta):
        """Change layer by delta"""
        new_val = self.layer_slider.value + delta
        if 0 <= new_val < len(self.layers):
            self.layer_slider.value = new_val
    
    def update_visualization(self, change):
        """Update visualization for selected layer"""
        layer_idx = self.layer_slider.value
        layer = self.layers[layer_idx]
        
        # Update button states
        self.prev_btn.disabled = (layer_idx == 0)
        self.next_btn.disabled = (layer_idx == len(self.layers) - 1)
        
        # Update info
        features_html = ''.join([f'<span style="background: {layer["color"]}22; padding: 3px 8px; '
                                 f'border-radius: 12px; margin: 2px; display: inline-block; '
                                 f'border: 1px solid {layer["color"]};">{f}</span>' 
                                 for f in layer['example_features']])
        
        self.layer_info.value = f"""
        <div style="background: linear-gradient(135deg, {layer['color']}15 0%, {layer['color']}30 100%); 
                    padding: 20px; border-radius: 12px; margin: 10px 0;
                    border-left: 5px solid {layer['color']};">
            <h3 style="margin: 0 0 10px 0; color: #333;">
                <span style="font-size: 28px;">{layer['icon']}</span> {layer['name']}
            </h3>
            <p style="color: #555; font-size: 15px; margin: 10px 0;">{layer['description']}</p>
            <p style="margin: 10px 0 5px 0;"><strong>What this layer detects:</strong></p>
            <div>{features_html}</div>
        </div>
        """
        
        with self.plot_output:
            clear_output(wait=True)
            self._create_full_visualization(layer_idx)
    
    def _create_full_visualization(self, active_idx):
        """Create the full pipeline visualization with active layer highlighted"""
        
        fig = make_subplots(
            rows=2, cols=1,
            row_heights=[0.35, 0.65],
            vertical_spacing=0.08,
            subplot_titles=(
                '<b>CNN Pipeline: Click slider to explore each stage</b>',
                '<b>What the Network "Sees" at This Layer</b>'
            )
        )
        
        # === ROW 1: Pipeline overview ===
        n_layers = len(self.layers)
        
        # Draw connections first
        for i in range(n_layers - 1):
            x1 = i
            x2 = i + 1
            opacity = 0.8 if (i == active_idx or i + 1 == active_idx) else 0.2
            
            fig.add_trace(go.Scatter(
                x=[x1 + 0.15, x2 - 0.15],
                y=[0, 0],
                mode='lines',
                line=dict(color='#666', width=3, dash='solid' if opacity > 0.5 else 'dot'),
                opacity=opacity,
                showlegend=False,
                hoverinfo='skip'
            ), row=1, col=1)
            
            # Arrow
            fig.add_annotation(
                x=x2 - 0.18, y=0,
                ax=x2 - 0.35, ay=0,
                xref='x', yref='y',
                axref='x', ayref='y',
                showarrow=True,
                arrowhead=2,
                arrowsize=1.5,
                arrowwidth=2,
                arrowcolor='#666' if opacity > 0.5 else '#ccc',
                row=1, col=1
            )
        
        # Draw layer nodes
        for i, layer in enumerate(self.layers):
            is_active = (i == active_idx)
            is_complete = (i < active_idx)
            
            size = 55 if is_active else 40
            opacity = 1.0 if is_active else (0.7 if is_complete else 0.3)
            
            # Glow effect for active
            if is_active:
                fig.add_trace(go.Scatter(
                    x=[i], y=[0],
                    mode='markers',
                    marker=dict(size=size + 20, color=layer['color'], opacity=0.3),
                    showlegend=False,
                    hoverinfo='skip'
                ), row=1, col=1)
            
            # Main node
            fig.add_trace(go.Scatter(
                x=[i], y=[0],
                mode='markers+text',
                marker=dict(
                    size=size,
                    color=layer['color'],
                    opacity=opacity,
                    line=dict(color='white', width=3 if is_active else 1)
                ),
                text=[layer['icon']],
                textfont=dict(size=20 if is_active else 16),
                textposition='middle center',
                showlegend=False,
                hoverinfo='text',
                hovertext=layer['name']
            ), row=1, col=1)
            
            # Label below
            fig.add_annotation(
                x=i, y=-0.35,
                text=f'<b>{layer["short"]}</b>',
                showarrow=False,
                font=dict(
                    size=12 if is_active else 10,
                    color='#333' if is_active else '#999'
                ),
                row=1, col=1
            )
        
        # === ROW 2: Feature visualization ===
        layer = self.layers[active_idx]
        
        if active_idx < len(self.layers) - 1:  # Not output layer
            try:
                img, colorscale = layer['visual']()
                
                fig.add_trace(go.Heatmap(
                    z=img[::-1] if len(img.shape) == 2 else img,
                    colorscale=colorscale,
                    showscale=False,
                    hovertemplate='Value: %{z:.0f}<extra></extra>'
                ), row=2, col=1)
            except:
                # Fallback visualization
                img = np.random.rand(20, 20) * 255
                fig.add_trace(go.Heatmap(
                    z=img,
                    colorscale='Viridis',
                    showscale=False
                ), row=2, col=1)
        else:
            # Output layer - show bar chart
            classes = ['Cat', 'Dog', 'Bird']
            probs = [95, 3, 2]
            colors = ['#4CAF50', '#FFC107', '#FF5722']
            
            fig.add_trace(go.Bar(
                x=classes,
                y=probs,
                marker_color=colors,
                text=[f'{p}%' for p in probs],
                textposition='outside',
                textfont=dict(size=16)
            ), row=2, col=1)
            
            fig.update_yaxes(title='Probability (%)', range=[0, 110], row=2, col=1)
        
        # Layout
        fig.update_layout(
            height=650,
            width=1100,
            showlegend=False,
            paper_bgcolor='white',
            plot_bgcolor='rgba(248, 249, 250, 0.5)',
            margin=dict(l=40, r=40, t=60, b=40)
        )
        
        fig.update_xaxes(visible=False, range=[-0.5, n_layers - 0.5], row=1, col=1)
        fig.update_yaxes(visible=False, range=[-0.6, 0.4], row=1, col=1)
        
        if active_idx < len(self.layers) - 1:
            fig.update_xaxes(visible=False, row=2, col=1)
            fig.update_yaxes(visible=False, row=2, col=1)
        
        fig.show()

# Create the explorer
hierarchy_explorer = CNNFeatureHierarchyExplorer()

display(HTML("""
<div class="insight-box">
    <h3 style="margin-top:0; color:#856404;">💡 The Hierarchy Principle</h3>
    <ul style="margin-bottom: 0;">
        <li><strong>Composition:</strong> Complex features are built from simpler ones</li>
        <li><strong>Increasing abstraction:</strong> From concrete (edges) to abstract (objects)</li>
        <li><strong>Translation invariance:</strong> A cat is a cat whether it's top-left or bottom-right</li>
        <li>This hierarchy mirrors how biological vision systems work!</li>
    </ul>
</div>
"""))